# Notebook 02: Data Preparation & Feature Engineering

## Objective
This notebook prepares the raw exchange rate and CPI datasets for econometric and machine learning modelling. The data will be cleaned, transformed, merged, and enriched with additional features before being saved as a processed dataset.

In [5]:
# import libraries
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from pathlib import Path


In [6]:
#create file paths
DATA_DIR = Path("../data/raw")

exchange_file = DATA_DIR / "HistoricalRateDetail.csv"
cpi_file = DATA_DIR / "EXCEL - CPI (COICOP 2018 - 8digit) (202604).xlsx"

In [7]:
#load raw datasets
exchange = pd.read_csv('D:/ERPT_ML_Research/data/raw/HistoricalRateDetail.csv')
cpi_coicop = pd.read_excel('D:/ERPT_ML_Research/data/raw/EXCEL - CPI (COICOP 2018 - 8digit) (202604).xlsx')

In [8]:
#verify the shape of the datasets
print("Exchange dataset shape:", exchange.shape)
print("COICOP dataset shape:", cpi_coicop.shape)

Exchange dataset shape: (2843, 2)
COICOP dataset shape: (391, 233)


In [ ]:
#inspect exchange dataset
exchange.head()


,Indicator,Description
0,Rand per US Dollar,Weighted average of the banks' daily rates at ...
1,Date,Value
2,2026-05-18,16.6453
3,2026-05-15,16.6666
4,2026-05-14,16.4063


### Clean Exchange Rate Data

The exchange rate dataset contains daily USD/ZAR observations. This section converts the date column into a datetime format, transforms the exchange rate into a numeric variable, and removes invalid records to ensure consistency.

In [10]:
#remove metadata rows
exchange.columns = exchange.iloc[1]

exchange = exchange.iloc[2:].reset_index(drop=True)

In [11]:
#inspect cleaned data
exchange.head()

1,Date,Value
0,2026-05-18,16.6453
1,2026-05-15,16.6666
2,2026-05-14,16.4063
3,2026-05-13,16.4604
4,2026-05-12,16.5431


In [12]:
#rename the columns
exchange.columns = ["Date", "ExchangeRate"]

In [13]:
#inspect changes
exchange.head()

,Date,ExchangeRate
0,2026-05-18,16.6453
1,2026-05-15,16.6666
2,2026-05-14,16.4063
3,2026-05-13,16.4604
4,2026-05-12,16.5431


In [14]:
#check data types
exchange.info()

<class 'pandas.DataFrame'>
RangeIndex: 2841 entries, 0 to 2840
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   Date          2841 non-null   str  
 1   ExchangeRate  2841 non-null   str  
dtypes: str(2)
memory usage: 44.5 KB


In [15]:
#convert data types
exchange["Date"] = pd.to_datetime(exchange["Date"])
exchange["ExchangeRate"] = pd.to_numeric(exchange["ExchangeRate"])

In [16]:
#verify the conversion
exchange.info()

<class 'pandas.DataFrame'>
RangeIndex: 2841 entries, 0 to 2840
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   Date          2841 non-null   datetime64[us]
 1   ExchangeRate  2841 non-null   float64       
dtypes: datetime64[us](1), float64(1)
memory usage: 44.5 KB


In [17]:
#check the date range
print("Start date:", exchange["Date"].min())
print("End date:", exchange["Date"].max())

Start date: 2015-01-02 00:00:00
End date: 2026-05-18 00:00:00


In [18]:
#sort chronologically
exchange = exchange.sort_values("Date").reset_index(drop=True)
exchange.head()

,Date,ExchangeRate
0,2015-01-02,11.6219
1,2015-01-05,11.7071
2,2015-01-06,11.7264
3,2015-01-07,11.7248
4,2015-01-08,11.6426


In [19]:
#convert daily exchange rates to monthly
exchange["YearMonth"] = exchange["Date"].dt.to_period("M")
exchange.head()

,Date,ExchangeRate,YearMonth
0,2015-01-02,11.6219,2015-01
1,2015-01-05,11.7071,2015-01
2,2015-01-06,11.7264,2015-01
3,2015-01-07,11.7248,2015-01
4,2015-01-08,11.6426,2015-01


In [20]:
#aggregate daily exchange rates to monthly averages
exchange_monthly = (
    exchange
    .groupby("YearMonth", as_index=False)
    .agg({"ExchangeRate": "mean"})
)

In [21]:
#convert the period back to a date
exchange_monthly["Date"] = exchange_monthly["YearMonth"].dt.to_timestamp()

In [22]:
#reorder the columns
exchange_monthly = exchange_monthly[
    ["Date", "ExchangeRate"]
]

In [23]:
#inspect result
exchange_monthly.head()

,Date,ExchangeRate
0,2015-01-01,11.566500
1,2015-02-01,11.577440
2,2015-03-01,12.068727
3,2015-04-01,12.012553
4,2015-05-01,11.969940


In [24]:
#verify transformation
print(exchange_monthly.shape)
exchange_monthly.info()

(137, 2)
<class 'pandas.DataFrame'>
RangeIndex: 137 entries, 0 to 136
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   Date          137 non-null    datetime64[us]
 1   ExchangeRate  137 non-null    float64       
dtypes: datetime64[us](1), float64(1)
memory usage: 2.3 KB


In [25]:
#create copy of COICOP CPI Dataset
cpi_coicop = cpi_coicop.copy()


In [26]:
#inspect the COICOP CPI Dataset
cpi_coicop.info()

<class 'pandas.DataFrame'>
RangeIndex: 391 entries, 0 to 390
Columns: 233 entries, Division to M202604
dtypes: float64(221), int64(6), str(6)
memory usage: 711.9 KB


In [27]:
cpi_coicop.head()

,Division,DivisionDescription,Group,GroupDescription,Class,ClassDescription,Subclass,SubclassDescription,Old code,Eight digit code,...,M202507,M202508,M202509,M202510,M202511,M202512,M202601,M202602,M202603,M202604
0,1,Food and non-alcoholic beverages,11,Food,111,Cereal products,1111,Cereals,1111001.0,1111201,...,96.9,95.2,94.6,94.1,91.2,91.6,88.3,87.7,87.1,86.0
1,1,Food and non-alcoholic beverages,11,Food,111,Cereal products,1111,Cereals,NaN,1111202,...,93.1,93.2,93.8,94.3,93.8,92.6,91.9,91.0,92.1,91.9
2,1,Food and non-alcoholic beverages,11,Food,111,Cereal products,1112,Flour of cereals,1116001.0,1112101,...,101.4,101.8,101.9,101.5,101.4,100.9,101.4,100.8,101.3,101.0
3,1,Food and non-alcoholic beverages,11,Food,111,Cereal products,1112,Flour of cereals,1116002.0,1112102,...,102.1,100.8,102.7,100.9,103.1,100.0,99.7,99.6,99.5,99.6
4,1,Food and non-alcoholic beverages,11,Food,111,Cereal products,1112,Flour of cereals,NaN,1112301,...,99.7,101.2,102.0,102.6,103.4,103.2,103.0,103.4,103.6,104.3


In [28]:
cpi_coicop["DivisionDescription"].value_counts()

DivisionDescription
Food and non-alcoholic beverages                            134
Clothing and footwear                                        47
Furnishings, household equipment and routine maintenance     41
Transport                                                    32
Housing and utilities                                        23
Health                                                       22
Recreation, sport and culture                                22
Personal care and miscellaneous services                     22
Information and communication                                16
Alcoholic beverages and tobacco                              14
Education services                                            6
Restaurants and accommodation services                        6
Insurance and financial services                              6
Name: count, dtype: int64

In [29]:
#filter dataset to Food and non-alcoholic beverages
food_cpi = cpi_coicop[
    cpi_coicop["DivisionDescription"] == "Food and non-alcoholic beverages"].copy()

In [30]:
food_cpi.shape

(134, 233)

In [31]:
#inspect filtered dataset
food_cpi.head()

,Division,DivisionDescription,Group,GroupDescription,Class,ClassDescription,Subclass,SubclassDescription,Old code,Eight digit code,...,M202507,M202508,M202509,M202510,M202511,M202512,M202601,M202602,M202603,M202604
0,1,Food and non-alcoholic beverages,11,Food,111,Cereal products,1111,Cereals,1111001.0,1111201,...,96.9,95.2,94.6,94.1,91.2,91.6,88.3,87.7,87.1,86.0
1,1,Food and non-alcoholic beverages,11,Food,111,Cereal products,1111,Cereals,NaN,1111202,...,93.1,93.2,93.8,94.3,93.8,92.6,91.9,91.0,92.1,91.9
2,1,Food and non-alcoholic beverages,11,Food,111,Cereal products,1112,Flour of cereals,1116001.0,1112101,...,101.4,101.8,101.9,101.5,101.4,100.9,101.4,100.8,101.3,101.0
3,1,Food and non-alcoholic beverages,11,Food,111,Cereal products,1112,Flour of cereals,1116002.0,1112102,...,102.1,100.8,102.7,100.9,103.1,100.0,99.7,99.6,99.5,99.6
4,1,Food and non-alcoholic beverages,11,Food,111,Cereal products,1112,Flour of cereals,NaN,1112301,...,99.7,101.2,102.0,102.6,103.4,103.2,103.0,103.4,103.6,104.3


In [32]:
#check food categories
food_cpi["GroupDescription"].value_counts()

GroupDescription
Food                       124
Non-alcoholic beverages     10
Name: count, dtype: int64

In [33]:
#identify monthly columns
monthly_columns = [
    col for col in food_cpi.columns
    if col.startswith("M")
]

print(f"Number of monthly columns: {len(monthly_columns)}")
monthly_columns[:5]

Number of monthly columns: 220


['M200801', 'M200802', 'M200803', 'M200804', 'M200805']

In [34]:
#choose identifier columns
id_columns = [
    "DivisionDescription",
    "GroupDescription",
    "ClassDescription",
    "SubclassDescription",
    "Weight",
    "Eight digit code"
]

In [35]:
#reshape dataset
food_cpi_long = food_cpi.melt(
    id_vars = id_columns,
    value_vars=monthly_columns,
    var_name="Month",
    value_name="CPI"
)

In [36]:
#inspect the result
food_cpi_long.head()

,DivisionDescription,GroupDescription,ClassDescription,SubclassDescription,Weight,Eight digit code,Month,CPI
0,Food and non-alcoholic beverages,Food,Cereal products,Cereals,0.540243,1111201,M200801,36.4
1,Food and non-alcoholic beverages,Food,Cereal products,Cereals,0.013386,1111202,M200801,NaN
2,Food and non-alcoholic beverages,Food,Cereal products,Flour of cereals,0.191734,1112101,M200801,36.4
3,Food and non-alcoholic beverages,Food,Cereal products,Flour of cereals,0.023029,1112102,M200801,41.9
4,Food and non-alcoholic beverages,Food,Cereal products,Flour of cereals,0.015759,1112301,M200801,NaN


In [37]:
#reshape the data
food_cpi_long = food_cpi.melt(
    id_vars=[
        "DivisionDescription",
        "GroupDescription",
        "ClassDescription",
        "SubclassDescription",
        "Weight",
        "Eight digit code"
    ],
    value_vars=monthly_columns,
    var_name="Month",
    value_name="CPI"
)

In [38]:
#inspect
food_cpi_long.shape

(29480, 8)

In [39]:
#view
food_cpi_long.head()

,DivisionDescription,GroupDescription,ClassDescription,SubclassDescription,Weight,Eight digit code,Month,CPI
0,Food and non-alcoholic beverages,Food,Cereal products,Cereals,0.540243,1111201,M200801,36.4
1,Food and non-alcoholic beverages,Food,Cereal products,Cereals,0.013386,1111202,M200801,NaN
2,Food and non-alcoholic beverages,Food,Cereal products,Flour of cereals,0.191734,1112101,M200801,36.4
3,Food and non-alcoholic beverages,Food,Cereal products,Flour of cereals,0.023029,1112102,M200801,41.9
4,Food and non-alcoholic beverages,Food,Cereal products,Flour of cereals,0.015759,1112301,M200801,NaN


In [40]:
#remove month labels to dates
food_cpi_long["Month"] = (
    food_cpi_long["Month"]
    .str.replace("M", "", regex=False)
)


In [41]:
#inspect
food_cpi_long.head()

,DivisionDescription,GroupDescription,ClassDescription,SubclassDescription,Weight,Eight digit code,Month,CPI
0,Food and non-alcoholic beverages,Food,Cereal products,Cereals,0.540243,1111201,200801,36.4
1,Food and non-alcoholic beverages,Food,Cereal products,Cereals,0.013386,1111202,200801,NaN
2,Food and non-alcoholic beverages,Food,Cereal products,Flour of cereals,0.191734,1112101,200801,36.4
3,Food and non-alcoholic beverages,Food,Cereal products,Flour of cereals,0.023029,1112102,200801,41.9
4,Food and non-alcoholic beverages,Food,Cereal products,Flour of cereals,0.015759,1112301,200801,NaN


In [42]:
#convert to datetime
food_cpi_long["Date"] = pd.to_datetime(
   food_cpi_long["Month"],
   format="%Y%m" 
)

In [43]:
#remove the old month column
food_cpi_long = food_cpi_long.drop(columns="Month")

In [44]:
#reorder the columns
food_cpi_long = food_cpi_long[
[
    "Date",
        "DivisionDescription",
        "GroupDescription",
        "ClassDescription",
        "SubclassDescription",
        "Weight",
        "Eight digit code",
        "CPI",
]
]

In [45]:
#inspect the result
food_cpi_long.head()

,Date,DivisionDescription,GroupDescription,ClassDescription,SubclassDescription,Weight,Eight digit code,CPI
0,2008-01-01,Food and non-alcoholic beverages,Food,Cereal products,Cereals,0.540243,1111201,36.4
1,2008-01-01,Food and non-alcoholic beverages,Food,Cereal products,Cereals,0.013386,1111202,NaN
2,2008-01-01,Food and non-alcoholic beverages,Food,Cereal products,Flour of cereals,0.191734,1112101,36.4
3,2008-01-01,Food and non-alcoholic beverages,Food,Cereal products,Flour of cereals,0.023029,1112102,41.9
4,2008-01-01,Food and non-alcoholic beverages,Food,Cereal products,Flour of cereals,0.015759,1112301,NaN


In [46]:
food_cpi_long.info()

<class 'pandas.DataFrame'>
RangeIndex: 29480 entries, 0 to 29479
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Date                 29480 non-null  datetime64[us]
 1   DivisionDescription  29480 non-null  str           
 2   GroupDescription     29480 non-null  str           
 3   ClassDescription     29480 non-null  str           
 4   SubclassDescription  29480 non-null  str           
 5   Weight               29480 non-null  float64       
 6   Eight digit code     29480 non-null  int64         
 7   CPI                  22693 non-null  float64       
dtypes: datetime64[us](1), float64(2), int64(1), str(4)
memory usage: 1.8 MB


In [47]:
#filter to research period
food_cpi_long = food_cpi_long[
    (food_cpi_long["Date"] >= "2015-01-01") &
    (food_cpi_long["Date"] <= "2025-12-01")
]

In [48]:
#verify
print(food_cpi_long.shape)

print("\nStart date:", food_cpi_long["Date"].min())
print("End date:", food_cpi_long["Date"].max())

(17688, 8)

Start date: 2015-01-01 00:00:00
End date: 2025-12-01 00:00:00


In [49]:
#check missing values
food_cpi_long["CPI"].isna().sum()

np.int64(2851)

In [50]:
#check dates for exhange and food cpi datasets
print(exchange_monthly["Date"].min())
print(exchange_monthly["Date"].max())

print(food_cpi_long["Date"].min())
print(food_cpi_long["Date"].max())

2015-01-01 00:00:00
2026-05-01 00:00:00
2015-01-01 00:00:00
2025-12-01 00:00:00


### Merge Datasets

The monthly CPI data is merged with the monthly average USD/ZAR exchange rate using the Date column. This produces the final modelling dataset that combines food prices with exchange rate information.

In [51]:
#merge CPI with monthly exchange rates
merged_data = pd.merge(
    food_cpi_long,
    exchange_monthly,
    on="Date",
    how="left"
)

In [52]:
#inspect
merged_data.head()

,Date,DivisionDescription,GroupDescription,ClassDescription,SubclassDescription,Weight,Eight digit code,CPI,ExchangeRate
0,2015-01-01,Food and non-alcoholic beverages,Food,Cereal products,Cereals,0.540243,1111201,56.6,11.5665
1,2015-01-01,Food and non-alcoholic beverages,Food,Cereal products,Cereals,0.013386,1111202,NaN,11.5665
2,2015-01-01,Food and non-alcoholic beverages,Food,Cereal products,Flour of cereals,0.191734,1112101,55.1,11.5665
3,2015-01-01,Food and non-alcoholic beverages,Food,Cereal products,Flour of cereals,0.023029,1112102,63.0,11.5665
4,2015-01-01,Food and non-alcoholic beverages,Food,Cereal products,Flour of cereals,0.015759,1112301,NaN,11.5665


In [53]:
#verify merge
merged_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 17688 entries, 0 to 17687
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Date                 17688 non-null  datetime64[us]
 1   DivisionDescription  17688 non-null  str           
 2   GroupDescription     17688 non-null  str           
 3   ClassDescription     17688 non-null  str           
 4   SubclassDescription  17688 non-null  str           
 5   Weight               17688 non-null  float64       
 6   Eight digit code     17688 non-null  int64         
 7   CPI                  14837 non-null  float64       
 8   ExchangeRate         17688 non-null  float64       
dtypes: datetime64[us](1), float64(3), int64(1), str(4)
memory usage: 1.2 MB


In [54]:
#check for missing values
merged_data["ExchangeRate"].isna().sum()

np.int64(0)

### Save Processed Dataset

The cleaned and merged dataset is saved to the processed data folder. This dataset will be used throughout the feature engineering, econometric modelling, and machine learning notebooks.

In [55]:
#save processed dataset
merged_data.to_csv(
    "../data/processed/merged_data.csv",
    index = False
)

print("Save is successful.")

Save is successful.
